# S/F Proximity Effect

The **proximity effect** in superconductor/ferromagnet (S/F) heterostructures
describes how Cooper pairs leak from S into F. The exchange field in the
ferromagnet causes the pair amplitude to oscillate and decay — an FFLO-like
effect that is the origin of many striking phenomena in S/F physics, including
$T_c$ oscillations, $\pi$-junctions, and re-entrant superconductivity.

## FFLO-Like Oscillation in the Ferromagnet

In a normal metal ($E_{\text{ex}} = 0$), Cooper pairs decay monotonically
into the metal over the thermal length $\xi_N = \sqrt{\hbar D / 2\pi k_B T}$.

In a ferromagnet, the exchange field $E_{\text{ex}}$ splits the Fermi surface
for spin-up and spin-down electrons. A singlet pair entering F acquires a
finite center-of-mass momentum, causing it to **oscillate** while decaying:

$$
F(x) = F_0 \exp\!\left(-\frac{x}{\xi_F}\right)
\cos\!\left(\frac{x}{\xi_F}\right)
$$

The characteristic length scale is the **ferromagnetic coherence length**:

$$
\xi_F = \sqrt{\frac{\hbar D_F}{E_{\text{ex}}}}
$$

| Material | $E_{\text{ex}}$ (meV) | $\xi_F$ (nm) | Character |
|----------|----------------------:|-------------:|----------|
| PdNi     | 2–5                   | 5–10         | Weak FM  |
| CuNi     | 5–15                  | 3–5          | Weak FM  |
| Ni       | 150                   | 1.2          | Strong FM |
| Fe       | 256                   | 0.7          | Strong FM |
| Co       | 300                   | 0.6          | Strong FM |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 6, 500)  # in units of xi_F

# 0-phase (cosine) and π-phase (sine) pair amplitudes
F_zero = np.exp(-x) * np.cos(x)
F_pi   = np.exp(-x) * np.sin(x)
envelope = np.exp(-x)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, F_zero, 'b-', linewidth=2, label=r'0-phase: $e^{-x}\cos x$')
ax.plot(x, F_pi, 'r--', linewidth=2, label=r'$\pi$-phase: $e^{-x}\sin x$')
ax.plot(x, envelope, 'k:', linewidth=1, alpha=0.5, label='Decay envelope')
ax.plot(x, -envelope, 'k:', linewidth=1, alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel(r'$x / \xi_F$', fontsize=13)
ax.set_ylabel(r'$F(x) / F_0$', fontsize=13)
ax.set_title('Pair amplitude in the ferromagnet', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 6)
fig.tight_layout()
plt.show()

## Critical Temperature Suppression

The leakage of Cooper pairs from S into F weakens superconductivity in the
S layer, suppressing $T_c$. For an S/F bilayer, $T_c(d_F)$ oscillates
as a function of the ferromagnet thickness $d_F$, reflecting the
constructive/destructive interference of the oscillating pair amplitude
reflected back into S.

### Single-Mode Approximation (Buzdin, 1982)

In the simplest approximation:

$$
T_c(d_F) \approx T_{c0} \left[ 1 - \frac{\xi_S}{d_S}
\operatorname{Re}\!\left(
\frac{1 - e^{-2\kappa_F d_F}}{\kappa_F \xi_S}
\right) \right]
$$

with complex wavevector $\kappa_F = (1+i)/\xi_F$.

### Digamma Self-Consistency Equation

The rigorous formulation uses the digamma function $\psi(z)$ to sum over
Matsubara frequencies. The $T_c$ equation for an S/F bilayer is:

$$
\ln\!\left(\frac{T_{c0}}{T_c}\right)
= \operatorname{Re}\!\left[
\psi\!\left(\frac{1}{2} + \frac{\gamma\, K(T_c)\, T_{c0}}{2\pi T_c}
+ \lambda_{\text{dep}}\right)
- \psi\!\left(\frac{1}{2}\right)
\right]
$$

where:
- $\psi(z)$ is the digamma function (logarithmic derivative of $\Gamma$)
- $\gamma = \xi_S / d_S$ is the S-layer transparency parameter
- $K(T_c)$ is the proximity kernel (depends on geometry and phase)
- $\lambda_{\text{dep}}$ collects pair-breaking contributions (orbital, spin-orbit, etc.)

#### Proximity Kernels

The kernel $K$ encodes how the ferromagnet feeds back into the S layer.
With complex wavevector $q = (1+i)/\xi_F$:

| Phase | Kernel | Formula |
|-------|--------|---------|
| 0     | coth   | $K = q\, \coth(q\, d_F) / (1 + \gamma_B\, q\, \coth(q\, d_F))$ |
| $\pi$ | tanh   | $K = q\, \tanh(q\, d_F) / (1 + \gamma_B\, q\, \tanh(q\, d_F))$ |

The boundary resistance parameter $\gamma_B$ accounts for interface transparency.

### The Fominov Model (PRB 66, 014507, 2002)

Fominov, Chtchelkatchev & Golubov extended the theory to include:
- Finite interface resistance ($\gamma_B \neq 0$)
- Self-consistent treatment of both S and F layers
- Arbitrary F-layer thickness

Their equation modifies the effective pair-breaking parameter:

$$
\alpha_{\text{eff}} = \frac{\gamma\, K}{1 + \gamma_B\, K}
$$

This model is essential for quantitative comparison with experiment,
particularly for weak ferromagnets with $\gamma_B \sim 0.1{-}1.0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import supermag

nb = supermag.get_material("Nb")
d_F = np.linspace(0.5, 25.0, 200)

fig, ax = plt.subplots(figsize=(8, 5))

# Compare different ferromagnets
for fm_name, color in [("Ni", "#d62728"), ("Cu0.43Ni0.57", "#2ca02c"), ("Py", "#1f77b4")]:
    fm = supermag.get_material(fm_name)
    Tc = supermag.critical_temperature(
        Tc0=nb["Tc"], d_S=50.0, d_F_array=d_F,
        E_ex=fm["E_ex"], xi_S=nb["xi_S"], xi_F=fm["xi_F"],
        gamma=0.15,
    )
    ax.plot(d_F, Tc, color=color, linewidth=2, label=fm_name)

ax.axhline(nb["Tc"], ls="--", color="gray", linewidth=0.8, label=r"$T_{c0}$")
ax.set_xlabel(r"$d_F$ (nm)", fontsize=13)
ax.set_ylabel(r"$T_c$ (K)", fontsize=13)
ax.set_title(r"$T_c$ suppression in Nb/F bilayers", fontsize=14)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

## Key Physics Summary

| Phenomenon | Origin | Observable |
|------------|--------|------------|
| Pair oscillation | Exchange-induced FFLO momentum | LDOS oscillation in STM |
| $T_c$ suppression | Pair leakage from S→F | Resistive transition |
| $T_c$ oscillation | Interference of reflected pairs | Non-monotonic $T_c(d_F)$ |
| Re-entrant SC | Destructive interference at specific $d_F$ | $T_c \to 0$ then recovery |
| $\pi$-state | Sign change of pair amplitude | Half-quantum flux in SQUIDs |

## References

1. Buzdin, A.I. et al., "Critical-current oscillations as a function of the exchange field and thickness of the ferromagnetic metal (F) in an S-F-S Josephson junction," JETP Lett. **35**, 178 (1982).
2. Radović, Z. et al., "Transition temperatures of superconductor-ferromagnet superlattices," Phys. Rev. B **44**, 759 (1991).
3. Fominov, Ya.V., Chtchelkatchev, N.M. & Golubov, A.A., "Nonmonotonic critical temperature in superconductor/ferromagnet bilayers," Phys. Rev. B **66**, 014507 (2002).
4. Buzdin, A.I., "Proximity effects in superconductor-ferromagnet heterostructures," Rev. Mod. Phys. **77**, 935 (2005).